# Gemma Model Test Notebook
This notebook tests the document retrieval and the local Gemma LLM generation pipeline.

In [3]:
!pip install -r requirements.txt

  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
  Using cached markdown_it_py-4.0.0-py3-none-any.whl.metadata (7.3 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 13.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 14.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.8/17.8 MB 13.0 MB/s eta 0:00:00a 0:00:01
   ━━

In [1]:
# Import necessary functions
from retriever import retrieve_documents
from llm import call_llm


## 1. Test Document Retrieval
First, we'll test retrieving documents using a sample query. Make sure you have ingested some PDFs using `vector_creator.py` beforehand.

In [2]:
query = "What is the main topic of the document?"

print(f"Retrieving documents for query: '{query}'")
retrieved_contents = retrieve_documents(query)

print("\n--- Retrieved Content ---")
print(retrieved_contents)

Retrieving documents for query: 'What is the main topic of the document?'
Loading vector database from 'chroma_db'...

Searching for: 'What is the main topic of the document?'

--- Top 1 Results ---

Result 1 (Source: data/Get_Started_With_Smallpdf.pdf, Page: 0):
Welcome to Smallpdf
Digital Documents—All In One Place
Access Files Anytime, Anywhere 
Enhance Documents in One Click 
Collaborate With Others 
With the new Smallpdf experience, you can 
freely upload, organize, and share digital 
documents. When you enable the ‘Storage’ 
option, we’ll also store all processed files here. 
You can access files stored on Smallpdf from 
your computer, phone, or tablet. We’ll also 
sync files from the Smallpdf Mobile App to our 
online portal
When you right-click on a file, we’ll present 
you with an array of options to convert, 
compress, or modify it. 
Forget mundane administrative tasks. With 
Smallpdf, you can request e-signatures, send 
large files, or even enable the Smallpdf G Suite 
App f

## 2. Test LLM Generation with Context (RAG)
Now we'll use the retrieved content to generate a response from the local Gemma model.

In [3]:
base_prompt = "Analyse the user query and retrieved contents and respond."
llm_input = f"{base_prompt}\n\nUser query: {query}\n\nRetrieved contents:\n{retrieved_contents}"

print("Calling LLM with RAG prompt...")
response = call_llm(llm_input)

Calling LLM with RAG prompt...
Sending request to local Ollama (Model: gemma4:26b)...
Waiting for response...

=== Response ===
The main topic of the document is an introduction to **Smallpdf**, an all-in-one digital document management platform. 

Specifically, the document outlines the platform's key features and capabilities, which include:
* **Document Management:** Uploading, organizing, and storing digital files for access across various devices (computer, phone, or tablet).
* **File Processing:** Tools to convert, compress, or modify documents with a single click.
* **Collaboration and Sharing:** Features for sharing files, requesting e-signatures, and collaborating with others.
* **Integration:** The ability to use the Smallpdf G Suite App for organizations.


## 3. Test LLM Generation without Context (Direct Prompt)
Let's test the LLM with a couple of direct prompts.

In [4]:
test_prompt_1 = "Explain the concept of 'zero-shot learning' in one simple sentence."
print("Testing Prompt 1...")
response_1 = call_llm(test_prompt_1)

Testing Prompt 1...
Sending request to local Ollama (Model: gemma4:26b)...
Waiting for response...

=== Response ===
Zero-shot learning is the ability of an AI to recognize and identify new objects or concepts that it has never explicitly encountered during its training.


In [5]:
test_prompt_2 = "What are the benefits of Retrieval-Augmented Generation (RAG)? Please list three points."
print("Testing Prompt 2...")
response_2 = call_llm(test_prompt_2)

Testing Prompt 2...
Sending request to local Ollama (Model: gemma4:26b)...
Waiting for response...

=== Response ===
Here are three key benefits of Retrieval-Augmented Generation (RAG):

1.  **Reduction of Hallucinations:** By providing the model with specific, factual context from external documents, RAG "grounds" the generation process. This significantly reduces the likelihood of the model fabricating information (hallucinating), as the model is instructed to base its answers on the retrieved data rather than relying solely on its internal weights.
2.  **Access to Up-to-Date and Private Information:** Standard Large Language Models (LLMs) are limited by their training cutoff date. RAG allows a model to access real-time information or proprietary, private datasets (such as company manuals or recent news) by simply updating the retrieval database, eliminating the need for expensive and time-consuming model retraining.
3.  **Improved Transparency and Traceability:** RAG enables "source